PATH PLANNING ALGORITHM BASED ON GENETIC AND HEURISTIC ALGORITHMS

Basic
Operations
for
Rover
Analysis
Software AKA B.O.R.A.S.


In [ ]:
##IMPORTS##

import numpy as np
import rasterio as rio
from rasterio.plot import show
import matplotlib.pyplot as plt
from time import perf_counter
import elevation 
import random
import copy
import datetime

Function Definitions:

In [ ]:

def move_position(grid_size,position,move,end):
    x, y = position
    if move[0] == 'U' and y > 0:
        new_position = (x,y-1)
    elif move[0] == 'D' and y < grid_size[1]-1:
        new_position = (x,y+1)
    elif move[0] == 'L' and x > 0:
        new_position = (x-1,y)
    elif move[0] == 'R' and x < grid_size[0]-1:
        new_position = (x+1,y)
    elif x == end[0] and y == end[1]:
        new_position = position
    else:
        new_position = position
    return new_position

In [ ]:
def evaluate_move(grid_size,move,start,end,position):
    position = move_position(grid_size,position,move,end)
    distance_to_end=abs(position[0]-end[0]) + abs(position[1]-end[1]) #Manhattan Distance
    max_dist = abs(start[0]-end[0]) + abs(start[1]-end[1]) #Manhattan Distance
    norm_dist = (max_dist - distance_to_end)/max_dist #Normalise
    return norm_dist

In [ ]:
def new_move(moves,start, end, grid_size,position):
    fitness = []
    for move in moves:
       total_distance = evaluate_move(grid_size,move,start,end,position) 
       fitness.append(total_distance)
    max_fitness = np.max(fitness)
    move = moves[fitness.index(max_fitness)]
    move = str(move).strip('[]')
    move = str(move).replace("'","")
    fitness= fitness[moves.index(move)]
    return move

In [ ]:
def create_segment(moves,start, end, grid_size,position):
    path = []
    position = start
    path.append(position)
    while position != end:
        move= new_move(moves, start, end, grid_size)
        position = tuple(move_position(grid_size,position,move,end))
        path.append(position)
    return path


In [ ]:
def create_path(start,end,moves, grid_size, num_points= 4):
    int_points = []
    path = []
    int_points.append(start)
    i = start[0]
    j = start[1]
    for _ in range(num_points):
        i = random.randint(0,grid_size[0])
        j = random.randint(0,grid_size[1])
        position = (i,j)
        int_points.append(position)
    int_points.append(end)
    for x in range(int_points[0]):
        segment = create_segment(moves,int_points[x],int_points[x+1],grid_size)
        path.extend(segment)
    return path



In [ ]:
def crossover(parent1,parent2,crossover_rate=0.9):
    if random.random()< crossover_rate:
        common_point1 =[]
        common_point2 =[]
        for x in parent1:
            for y in parent2:
                if x == y:
                    common_point1 = tuple(x)
                    common_point2 = tuple(y)
        if  not common_point1 or common_point2:
            return parent1
        else:
            crossover_points= random.randint(0,len(common_point1))
            child = parent1[:parent1.index(common_point1[crossover_points])] + parent2[parent2.index(common_point2[crossover_points]):]
            return child
    else:
        return parent1